In [1]:
import torch
from training_functions import train, evaluate_mae_mse
from TransCrowd import base_patch16_384_token, base_patch16_384_gap
import torch.nn as nn
import torch.optim as optim
from dataloader_mcnn import CrowdDataset
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, ConcatDataset
import glob
import os
import numpy as np
import random
import time
import pandas as pd
from torchvision import transforms
SEED = 42

e:\OneDrive\Pulpit\MSC-thesis\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
e:\OneDrive\Pulpit\MSC-thesis\.venv\Lib\site-packages\timm\models\registry.py:4: FutureWarning: Importing from timm.models.registry is deprecated, please import via timm.models
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.models", FutureWarning)
e:\OneDrive\Pulpit\MSC-thesis\.venv\Lib\site-packages\timm\models\layers\__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


In [ ]:

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model = base_patch16_384_gap(pretrained=True, path = r'output\models\transcrowd\deit_base_patch16_384-8de9b5d1.pth')
model.load_state_dict(torch.load(r'output\models\transcrowd\transcrowd.pth'))
model.eval()
model = model.to(device)

### General tests

In [ ]:
transform = transforms.Compose([
                                transforms.ToTensor(),

                                transforms.Normalize(mean=[0.485, 0.456, 0.406],
                                                     std=[0.229, 0.224, 0.225]),
                            ])

main_folders = main_folders = [r"data\ShanghaiTech_part_A\test", 
                r"data\ShanghaiTech_part_B\test",
               r'data\NWPU-Crowd\val',
              r"data\UCF-QNRF_ECCV18\test",
               r"data\jhu_crowd_v2.0\test",
               r"data\UCF_CC_50"]
evaluations = []
for folder in main_folders:    
    test_img_root = os.path.join(folder, "images")
    test_map_root = os.path.join(folder, "maps")
    test_dataset = CrowdDataset(test_img_root, test_map_root, gt_downsample=1, resize=False, transform=transform, patch=True)
    test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False, pin_memory=True)
    mae, mse = evaluate_mae_mse(model, test_loader, device, is_dmap=False)
    rmse = mse**(1/2)
    evaluations.append({'dataset': folder.split('/')[-2] ,'mae': mae, 'rmse': rmse})

In [ ]:
df2 = pd.DataFrame(evaluations)
df2.to_csv(r"output\history\transcrowd\evaluation.csv")

### Advanced tests

In [ ]:
transform = transforms.Compose([
                                transforms.ToTensor(),

                                transforms.Normalize(mean=[0.485, 0.456, 0.406],
                                                     std=[0.229, 0.224, 0.225]),
                            ])
main_folders = main_folders = [r"data\ShanghaiTech_part_A\test", 
                r"data\ShanghaiTech_part_B\test",
               r'data\NWPU-Crowd\val',
              r"data\UCF-QNRF_ECCV18\test",
               r"data\jhu_crowd_v2.0\test",
               r"data\UCF_CC_50"]
evaluations = []
model.eval()
with torch.no_grad():
    for folder in main_folders:    
        test_img_root = os.path.join(folder, "images")
        test_map_root = os.path.join(folder, "maps")
        test_dataset = CrowdDataset(test_img_root, test_map_root, gt_downsample=1, resize=False, transform=transform, patch=True)
        for _,img,gt_dmap in test_dataset:
            img = img.to(device)
            pred_sum = model(img).sum().item()
            del img
            torch.cuda.empty_cache()
            gt = gt_dmap.sum().item()
            diff = pred_sum - gt
            diff_abs = abs(diff)
            perc = diff_abs / (gt+1e-9)
            evaluations.append({'dataset': folder.split('/')[-2] ,'pred': pred_sum, 'gt': gt, 'diff': diff, 'abs_err': diff_abs, 'perc': perc})

In [ ]:
df2 = pd.DataFrame(evaluations)
df2.to_csv(r"output\history\transcrowd\evaluation_extended.csv")

In [ ]:
df2 = pd.read_csv(r'output\history\transcrowd\evaluation_extended.csv')
df2 = df2[df2['gt'] != 0]
df2.groupby('dataset').mean()

In [ ]:
plt.scatter(df2['pred'], df2['diff'])
plt.title('TransCrowd')
plt.xlabel('Predicted')
plt.ylabel('Difference')

In [ ]:
plt.scatter(df2['gt'], df2['pred'])
plt.plot([0,max(df2['gt'])], [0, max(df2['gt'])], color='red')
plt.title('TransCrowd')
plt.xlabel('Ground truth')
plt.ylabel('Predicted')

In [ ]:
plt.scatter(df2['gt'], df2['pred'])
plt.xscale('log')
plt.yscale('log')

m = max(df2['gt'].max(), df2['pred'].max())
plt.plot([1, m], [1, m], color='red')

plt.xlabel('Ground truth')
plt.ylabel('Predicted')
plt.title('TransCrowd')

In [ ]:
df2 = pd.read_csv(r'output\history\transcrowd\evaluation_extended.csv')
df2 = df2[df2['gt'] == 0]

In [ ]:
bins = np.logspace(
    0,
    np.log10(df2['pred'].max()),
    15
)

plt.hist(
    df2['pred'],
    bins=bins,
    edgecolor='black',
    linewidth=1.2
)

plt.xscale('log')

plt.xlabel("Log Prediction")
plt.ylabel("Count")
plt.title("TransCrowd (Images with zero ground truth)")

plt.show()

In [ ]:
def bins(x):
    if x <= 50:
        return 'small'
    if x <= 200:
        return 'medium'
    if x <= 1000:
        return 'high'
    else:
        return 'large'

In [ ]:
df2 = pd.read_csv(r'output\history\transcrowd\evaluation_extended.csv')
df2 = df2[df2['gt'] != 0]
df2['bins'] = df2['gt'].apply(lambda x: bins(x))
df2['rmse'] = (df2['pred'] - df2['gt'])**2
df2.drop(columns='dataset', inplace=True)
df2 = df2.groupby(['bins']).mean()
df2['rmse'] = df2['rmse']**(1/2)
df2

### Finetuned tests

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model = base_patch16_384_gap(pretrained=True, path = r'output\models\transcrowd\deit_base_patch16_384-8de9b5d1.pth')
model.load_state_dict(torch.load(r'output\models\transcrowd\transcrowd_finetuned.pth'))
model.eval()
model = model.to(device)

In [ ]:
transform = transforms.Compose([
                                transforms.ToTensor(),

                                transforms.Normalize(mean=[0.485, 0.456, 0.406],
                                                     std=[0.229, 0.224, 0.225]),
                            ])
main_folders = main_folders = [r"data\ShanghaiTech_part_A\test", 
                r"data\ShanghaiTech_part_B\test",
               r'data\NWPU-Crowd\val',
              r"data\UCF-QNRF_ECCV18\test",
               r"data\jhu_crowd_v2.0\test",
               r"data\UCF_CC_50"]
evaluations = []
model.eval()
with torch.no_grad():
    for folder in main_folders:    
        test_img_root = os.path.join(folder, "images")
        test_map_root = os.path.join(folder, "maps")
        test_dataset = CrowdDataset(test_img_root, test_map_root, gt_downsample=1, resize=False, transform=transform, patch=True)
        for _,img,gt_dmap in test_dataset:
            img = img.to(device)
            pred_sum = model(img).sum().item()
            del img
            torch.cuda.empty_cache()
            gt = gt_dmap.sum().item()
            diff = pred_sum - gt
            diff_abs = abs(diff)
            perc = diff_abs / (gt+1e-9)
            evaluations.append({'dataset': folder.split('/')[-2] ,'pred': pred_sum, 'gt': gt, 'diff': diff, 'abs_err': diff_abs, 'perc': perc})

In [ ]:
df2 = pd.DataFrame(evaluations)
df2.to_csv(r"output\history\transcrowd\evaluation_extended_finetuned.csv")

In [ ]:
df3 = pd.read_csv(r'output\history\transcrowd\evaluation_extended_finetuned.csv')
df3.drop(columns='Unnamed: 0', inplace=True)
df3 = df3[df3['gt'] != 0]
df3.groupby('dataset').mean()

In [ ]:
plt.scatter(df3['pred'], df3['diff'])
plt.title('TransCrowd finetuned')
plt.xlabel('Predicted')
plt.ylabel('Difference')

In [ ]:
plt.scatter(df3['gt'], df3['pred'])
plt.plot([0,max(df3['gt'])], [0, max(df3['gt'])], color='red')
plt.title('TransCrowd finetuned')
plt.xlabel('Ground truth')
plt.ylabel('Predicted')

In [ ]:
plt.scatter(df3['gt'], df3['pred'])
plt.xscale('log')
plt.yscale('log')

m = max(df3['gt'].max(), df3['pred'].max())
plt.plot([1, m], [1, m], color='red')

plt.xlabel('Ground truth')
plt.ylabel('Predicted')
plt.title('TransCrowd finetuned')

In [ ]:
df3 = pd.read_csv(r'output\history\transcrowd\evaluation_extended_finetuned.csv')
df3 = df3[df3['gt'] == 0]

bins = np.logspace(
    0,
    np.log10(df3['pred'].max()),
    15
)

plt.hist(
    df3['pred'],
    bins=bins,
    edgecolor='black',
    linewidth=1.2
)

plt.xscale('log')

plt.xlabel("Log Prediction")
plt.ylabel("Count")
plt.title("TransCrowd finetuned (Images with zero ground truth)")

plt.show()

In [ ]:
def bins(x):
    if x <= 50:
        return 'small'
    if x <= 200:
        return 'medium'
    if x <= 1000:
        return 'high'
    else:
        return 'large'

In [ ]:
df3 = pd.read_csv(r'output\history\transcrowd\evaluation_extended_finetuned.csv')
df3 = df3[df3['gt'] != 0]
df3['bins'] = df3['gt'].apply(lambda x: bins(x))
df3['rmse'] = (df3['pred'] - df3['gt'])**2
df3.drop(columns='dataset', inplace=True)
df3 = df3.groupby(['bins']).mean()
df3['rmse'] = df3['rmse']**(1/2)
df3[['abs_err', 'rmse', 'perc']]